In [ ]:
# SPDX-License-Identifier: Apache-2.0 AND CC-BY-NC-4.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Solution to the UST Exercise

Some code is repeated here to make this notebook self-contained.

In [ ]:
import cupy as cp
import cupyx.scipy.sparse as sp
from cupyx.profiler import benchmark

import matplotlib.pyplot as plt

from nvmath.sparse.generic import Matmul
from nvmath.sparse.ust import NamedFormats, Tensor

n = 1024 * 32
values = cp.array([[0.0] + [-1.0] * (n - 1), [4.0] * n, [1.0] * (n - 1) + [0.0]], dtype=cp.float32)
offsets = cp.array([-1, 0, 1], dtype=cp.int32)
a_cp_dia = sp.dia_matrix((values, offsets), shape=(n, n))
a_cp_csr = sp.csr_matrix(a_cp_dia)
a_cp_coo = sp.coo_matrix(a_cp_dia)
a_cp_coo.sum_duplicates()   # coalesce COO format

cp_ones = cp.ones((n,), dtype=cp.float32)
cp_out = cp.zeros((n,), dtype=cp.float32)

u_dia = Tensor.from_package(a_cp_dia)
u_csr = Tensor.from_package(a_cp_csr)
u_coo = Tensor.from_package(a_cp_coo)
u_ones = Tensor.from_package(cp_ones)
u_out = Tensor.from_package(cp_out)

# SOLUTION
u_dcsr = u_coo.convert(tensor_format=NamedFormats.DCSR)


with Matmul(u_dia, u_ones, u_out) as mm:
    mm.plan()
    ust1 = benchmark(mm.execute, n_repeat=10)
    
with Matmul(u_csr, u_ones, u_out) as mm:
    mm.plan()
    ust2 = benchmark(mm.execute, n_repeat=10)    
    
with Matmul(u_coo, u_ones, u_out) as mm:
    mm.plan()
    ust3 = benchmark(mm.execute, n_repeat=10)

# SOLUTION
with Matmul(u_dcsr, u_ones, u_out) as mm:
    mm.plan()
    ust4 = benchmark(mm.execute, n_repeat=10)

print(ust1)
print(ust2)
print(ust3)   
print(ust4)   

# SOLUTION
labels = ["ust-dia", "ust-csr", "ust-coo", "ust-dcsr"]
runtimes = [
    min(ust1.gpu_times[0]),
    min(ust2.gpu_times[0]),
    min(ust3.gpu_times[0]),
    min(ust4.gpu_times[0]),
]
bar_colors = ["#76B900", "#76B900", "#76B900", "#76B900"]
hatch = ["+", ".", "/", "x"]
plt.figure(figsize=(8, 6))
plt.bar(labels, runtimes, color=bar_colors, hatch=hatch)
plt.ylabel("Execution Time (us) (LOG)")
plt.xticks(rotation=45)
plt.show()